# Week 5 Exercise — Simple RAG Knowledge Worker

A **RAG** (Retrieval-Augmented Generation) Q&A assistant: retrieve relevant context from a small knowledge base, then generate answers with an LLM. Uses **keyword retrieval** (like course Day 1) and **OpenRouter** (and optional Ollama) so it runs without embeddings or Chroma.

Set `OPENROUTER_API_KEY` in `.env`. For Ollama: `ollama pull llama3.2` and have Ollama running.

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
load_dotenv(override=True)
api_key = os.getenv("OPENROUTER_API_KEY")

if not api_key:
    print("Add OPENROUTER_API_KEY to .env")
else:
    print("OpenRouter API key loaded.")

MODELS = {
    "OpenRouter: gpt-4o-mini": ("openai/gpt-4o-mini", "https://openrouter.ai/api/v1", api_key),
    "OpenRouter: claude-3-haiku": ("anthropic/claude-3-haiku", "https://openrouter.ai/api/v1", api_key),
    "Ollama: llama3.2": ("llama3.2", "http://localhost:11434/v1", "ollama"),
}

def get_client(model_key):
    model_id, base_url, key = MODELS.get(model_key, list(MODELS.values())[0])
    return OpenAI(api_key=key or "ollama", base_url=base_url), model_id

In [ ]:
# Small in-notebook knowledge base (key phrases -> document text)
KNOWLEDGE = {
    "product": "Product CarLLM: A car insurance product that uses LLMs to assess driver behavior and offer personalized premiums. Launched in 2024. Key features: telematics integration, real-time risk scoring.",
    "team": "The Insurellm team includes Alex Lancaster (CEO), engineering in London and Lagos, and a focus on low-cost, accurate AI for insurance.",
    "company": "Insurellm is an Insurance Tech company. Expert knowledge worker assistants use RAG for accuracy. Solutions are designed to be low cost.",
    "award": "Alex Lancaster won the IIOTY (Insurance Innovator of the Year) award. Manchester University was mentioned in relation to team education.",
}

def get_relevant_context(message):
    text = "".join(c for c in message if c.isalpha() or c.isspace())
    words = set(w.lower() for w in text.split() if len(w) > 2)
    relevant = []
    for key, doc in KNOWLEDGE.items():
        if any(w in doc.lower() for w in words) or any(w in key for w in words):
            relevant.append(doc)
    if not relevant:
        return "No specific context found for this question. Answer from general knowledge or say you don't know."
    return "Relevant context from the knowledge base:\n\n" + "\n\n".join(relevant)

In [ ]:
SYSTEM_PREFIX = """You represent Insurellm, an Insurance Tech company.
You are an expert knowledge worker. Use ONLY the provided context to answer. Be brief and accurate.
If the context doesn't contain the answer, say so. Do not make up facts.

"""

def chat(message, history, model_key):
    client, model_id = get_client(model_key)
    context = get_relevant_context(message)
    system_message = SYSTEM_PREFIX + context
    history_fmt = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history_fmt + [{"role": "user", "content": message}]
    try:
        r = client.chat.completions.create(model=model_id, messages=messages)
        return r.choices[0].message.content or ""
    except Exception as e:
        return f"Error: {e}"

In [ ]:
model_dd = gr.Dropdown(choices=list(MODELS.keys()), value=list(MODELS.keys())[0], label="Model")
gr.ChatInterface(
    fn=chat,
    additional_inputs=[model_dd],
    type="messages",
    title="Week 5 — RAG Knowledge Worker",
    description="Ask about Insurellm: products, team, company, awards. Context is retrieved from a small KB and passed to the LLM.",
    examples=[
        ["What is CarLLM?", list(MODELS.keys())[0]],
        ["Who won the IIOTY award?", list(MODELS.keys())[0]],
    ],
).launch()